# GitHub Issues 多标签分类器

本练习使用 `goosmanlei/github-issues` 数据集，微调一个多标签文本分类模型，
根据 issue 的标题和正文自动预测它应该被打上哪些标签（bug、enhancement、documentation 等）。

**多标签分类**与多分类的区别：每个样本可以同时属于多个类别（one-hot 向量，而非单一整数标签）。

In [ ]:
!pip install datasets evaluate transformers[sentencepiece] accelerate scikit-learn -q

## 1. 加载数据集

In [1]:
from datasets import load_dataset

raw_dataset = load_dataset(
    "goosmanlei/github-issues",
    split="train",
    chunksize=30 << 20,
)
raw_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 7980
})

## 2. 数据探索与预处理

### 2.1 查看标签结构

In [2]:
# 查看一条有标签的样本
sample = next(x for x in raw_dataset if x["labels"])
print("title:", sample["title"])
print("labels:", sample["labels"])
print("is_pr:", sample["pull_request"] is not None)

title: Support 3D Datasets
labels: [{'id': 1935892871, 'node_id': 'MDU6TGFiZWwxOTM1ODkyODcx', 'url': 'https://api.github.com/repos/huggingface/datasets/labels/enhancement', 'name': 'enhancement', 'color': 'a2eeef', 'default': True, 'description': 'New feature or request'}]
is_pr: False


### 2.2 过滤 Pull Request，只保留 Issues

In [3]:
# 去掉 PR（只保留真正的 issues）
issues_only = raw_dataset.filter(lambda x: x["pull_request"] is None)
print(f"全部条目: {len(raw_dataset)}  |  仅 Issues: {len(issues_only)}")

Filter:   0%|          | 0/7980 [00:00<?, ? examples/s]

全部条目: 7980  |  仅 Issues: 3302


### 2.3 统计最常见标签，选出 Top-K

In [4]:
from collections import Counter

label_counter = Counter()
for labels in issues_only["labels"]:
    for lbl in labels:
        label_counter[lbl["name"]] += 1

print("Top-20 标签:")
for name, count in label_counter.most_common(20):
    print(f"  {name:<30} {count}")

Top-20 标签:
  bug                            709
  enhancement                    502
  dataset request                161
  dataset-viewer                 91
  dataset bug                    74
  good first issue               53
  duplicate                      30
  documentation                  28
  generic discussion             27
  question                       24
  vision                         24
  streaming                      22
  speech                         19
  maintenance                    17
  nlp-viewer                     14
  good second issue              13
  hacktoberfest                  12
  metric bug                     8
  wontfix                        7
  Dataset discussion             6


In [5]:
# 选取出现次数 >= 30 的标签作为分类目标
MIN_COUNT = 30
TARGET_LABELS = [name for name, cnt in label_counter.items() if cnt >= MIN_COUNT]
TARGET_LABELS = sorted(TARGET_LABELS)  # 排序保证稳定性
NUM_LABELS = len(TARGET_LABELS)

print(f"目标标签数量: {NUM_LABELS}")
print(TARGET_LABELS)

目标标签数量: 7
['bug', 'dataset bug', 'dataset request', 'dataset-viewer', 'duplicate', 'enhancement', 'good first issue']


### 2.4 构建多标签 One-Hot 向量

In [6]:
label2id = {name: i for i, name in enumerate(TARGET_LABELS)}
id2label = {i: name for name, i in label2id.items()}

def encode_labels(example):
    """将 labels 列表转为 float 多热向量，同时拼接 title + body 作为输入文本。"""
    vec = [0.0] * NUM_LABELS
    for lbl in example["labels"]:
        idx = label2id.get(lbl["name"])
        if idx is not None:
            vec[idx] = 1.0
    
    body = example["body"] or ""
    # 截断 body 避免过长（最多 512 字符），title 作前缀
    text = example["title"] + " [SEP] " + body[:512]
    return {"text": text, "label_vec": vec}

# 只保留有至少一个目标标签的 issues
processed = issues_only.map(encode_labels, remove_columns=issues_only.column_names)
processed = processed.filter(lambda x: sum(x["label_vec"]) > 0)
print(f"有效样本数: {len(processed)}")

Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3302 [00:00<?, ? examples/s]

有效样本数: 1528


## 3. 数据集划分

In [7]:
split = processed.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
test_ds  = split["test"]
print(f"训练集: {len(train_ds)}  |  测试集: {len(test_ds)}")

训练集: 1222  |  测试集: 306


## 4. Tokenization

In [8]:
from transformers import AutoTokenizer

MODEL_CKPT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    tokens["labels"] = batch["label_vec"]
    return tokens

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text", "label_vec"])
test_tok  = test_ds.map(tokenize,  batched=True, remove_columns=["text", "label_vec"])

train_tok.set_format("torch")
test_tok.set_format("torch")

train_tok

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1222 [00:00<?, ? examples/s]

Map:   0%|          | 0/306 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 1222
})

## 5. 定义模型

使用 `problem_type="multi_label_classification"` 让 HuggingFace 自动使用 `BCEWithLogitsLoss`。

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)
print(f"分类头输出维度: {model.classifier.out_features}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


分类头输出维度: 7


## 6. 定义评估指标

多标签分类常用 **micro-F1**（汇总所有标签的 TP/FP/FN）和 **macro-F1**（各标签 F1 的均值）。

In [10]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # 用 sigmoid + 0.5 阈值将 logits 转为二值预测
    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    # 计算每个标签的 AUC（要求每列至少有一个正例）
    try:
        auc = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        auc = float("nan")

    return {"micro_f1": micro_f1, "macro_f1": macro_f1, "roc_auc": auc}

## 7. 训练

In [11]:
from transformers import TrainingArguments, Trainer

BATCH_SIZE = 16

training_args = TrainingArguments(
    output_dir="github-issues-classifier",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    logging_steps=50,
    warmup_ratio=0.1,
    fp16=True,          # 如果没有 GPU 请改为 False
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/bytedance/miniforge3/envs/llm-learn/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Roc Auc
1,0.533986,0.304358,0.549587,0.142320,0.741110
2,0.230238,0.205442,0.816189,0.387149,0.767681
3,0.177239,0.178910,0.837209,0.471615,0.787688
4,0.159135,0.173838,0.844007,0.496729,0.789522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/bytedance/miniforge3/envs/llm-learn/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/bytedance/miniforge3/envs/llm-learn/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=308, training_loss=0.2618583812342062, metrics={'train_runtime': 141.5452, 'train_samples_per_second': 34.533, 'train_steps_per_second': 2.176, 'total_flos': 323779190452224.0, 'train_loss': 0.2618583812342062, 'epoch': 4.0})

## 8. 评估

In [12]:
metrics = trainer.evaluate()
print(f"Micro-F1 : {metrics['eval_micro_f1']:.4f}")
print(f"Macro-F1 : {metrics['eval_macro_f1']:.4f}")
print(f"ROC-AUC  : {metrics['eval_roc_auc']:.4f}")

/Users/bytedance/miniforge3/envs/llm-learn/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Micro-F1 : 0.8440
Macro-F1 : 0.4967
ROC-AUC  : 0.7895


In [13]:
# 按标签查看详细 F1
import pandas as pd
from sklearn.metrics import classification_report

pred_output = trainer.predict(test_tok)
logits = pred_output.predictions
true_labels = pred_output.label_ids.astype(int)
pred_labels = (1 / (1 + np.exp(-logits)) >= 0.5).astype(int)

report = classification_report(
    true_labels, pred_labels,
    target_names=TARGET_LABELS,
    zero_division=0,
    output_dict=True,
)
df_report = pd.DataFrame(report).T
df_report[["precision", "recall", "f1-score", "support"]].round(3)

/Users/bytedance/miniforge3/envs/llm-learn/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


,precision,recall,f1-score,support
bug,0.833,0.940,0.883,133.0
dataset bug,0.000,0.000,0.000,25.0
dataset request,1.000,0.821,0.902,28.0
dataset-viewer,0.923,0.632,0.750,19.0
duplicate,0.000,0.000,0.000,5.0
enhancement,0.942,0.942,0.942,103.0
good first issue,0.000,0.000,0.000,7.0
micro avg,0.889,0.803,0.844,320.0
macro avg,0.528,0.476,0.497,320.0
weighted avg,0.792,0.803,0.794,320.0


## 9. 推理演示

In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval().to(device)

def predict_labels(title: str, body: str = "", threshold: float = 0.4):
    text = title + " [SEP] " + body[:512]
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=256, padding="max_length"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    probs = torch.sigmoid(logits).cpu().numpy()
    results = [
        {"label": id2label[i], "score": float(probs[i])}
        for i in range(NUM_LABELS) if probs[i] >= threshold
    ]
    return sorted(results, key=lambda x: -x["score"])


# 测试样例
test_cases = [
    ("TypeError when loading dataset with streaming=True",
     "I get a TypeError when I try to load the dataset in streaming mode."),
    ("Add support for Parquet v2 format",
     "It would be great to support reading Parquet files written with v2 encoding."),
    ("How to contribute a new dataset?",
     "I want to add a new benchmark dataset to the hub, what's the process?"),
]

for title, body in test_cases:
    preds = predict_labels(title, body)
    print(f"标题: {title}")
    for p in preds:
        print(f"  {p['label']:<30} {p['score']:.3f}")
    print()

标题: TypeError when loading dataset with streaming=True
  bug                            0.687

标题: Add support for Parquet v2 format
  enhancement                    0.892

标题: How to contribute a new dataset?
  enhancement                    0.647



## 10. （可选）推送模型到 Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

trainer.push_to_hub(
    commit_message="Fine-tuned DistilBERT for multi-label GitHub issue classification"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            